In [1]:
import pandas as pd
import numpy as np
import tkinter as tk

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from tkinter.filedialog import askopenfilename
from sklearn.metrics import confusion_matrix, classification_report

In [2]:
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

file_path = askopenfilename(parent=root, title="Select CSV or Excel file")

root.destroy()

if file_path:
    if file_path.lower().endswith(".csv"):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_excel(file_path, engine="openpyxl")
    #print(df.head())
else:
    print("No file selected.")
    
#df = pd.read_csv ('data/Customer_Loan_Request.csv')
#data = pd.read_csv ('data/Customer_Loan_Request.csv')

df.head (3) 

,loan_id,age,gender,marital_status,dependents,education,employment_status,employment_years,annual_income,credit_score,...,existing_loans_count,loan_purpose,loan_amount,loan_term_months,interest_rate,property_ownership,savings_balance,collateral_value,debt_to_income_ratio,loanapproval
0,LN100000,43,Male,Divorced,1,High School,Self-Employed,21.5,16354,595,...,2,Auto,4200,12,10.66,Mortgage,3079,4856,0.812,0
1,LN100001,36,Female,Married,2,Bachelor's,Employed,9.6,59460,579,...,0,Home,121000,36,12.09,Own,1758,155688,0.678,0
2,LN100002,45,Male,Married,1,Master's,Self-Employed,7.2,89708,574,...,1,Business,33200,180,10.98,Rent,2280,33927,0.037,1


In [3]:

X = df.drop(columns=["loanapproval"])
y = df["loanapproval"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
numeric_features = [
    "employment_years",
    "annual_income",
    "credit_score",
    "previous_default",	
    "existing_loans_count",	
    "loan_amount",	
    "loan_term_months",	
    "interest_rate"	
]

categorical_features = [
    "loan_id",
    "gender",
    "marital_status",
    "education",
    "employment_status",
    "loan_purpose",
    "property_ownership",
    "savings_balance",
    "collateral_value",
    "debt_to_income_ratio"
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


In [5]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    subsample=0.8,
    colsample_bytree=0.8,
    #early_stopping_rounds=20,
    eval_metric="auc"    
)



In [6]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

In [7]:
pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['employment_years',
                                                   'annual_income',
                                                   'credit_score',
                                                   'previous_default',
                                                   'existing_loans_count',
                                                   'loan_amount',
                                                   'loan_term_months',
                                                   'interest_rate']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent'...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=300, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [8]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['employment_years',
                                                   'annual_income',
                                                   'credit_score',
                                                   'previous_default',
                                                   'existing_loans_count',
                                                   'loan_amount',
                                                   'loan_term_months',
                                                   'interest_rate']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent'...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=300, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [9]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]


In [10]:
pd.DataFrame({"Actual_Loan": y_test, "Prediction": y_pred, "Probability": y_prob.round (2)})


,Actual_Loan,Prediction,Probability
3551,1,0,0.37
10404,1,1,0.77
455,1,1,0.67
7566,1,1,0.94
9653,1,1,0.76
...,...,...,...
1247,0,0,0.01
7255,0,0,0.01
9364,0,0,0.07
5756,0,0,0.11


In [11]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[760 233]
 [209 898]]
              precision    recall  f1-score   support

           0       0.78      0.77      0.77       993
           1       0.79      0.81      0.80      1107

    accuracy                           0.79      2100
   macro avg       0.79      0.79      0.79      2100
weighted avg       0.79      0.79      0.79      2100



In [12]:
import pandas as pd

def calculate_loan_decision(row):

    score = 0
    reasons = []

    # -------------------------------------------------------
    # 1. Credit Score (30 Points)
    # -------------------------------------------------------
    if row["credit_score"] >= 800:
        score += 30
    elif row["credit_score"] >= 750:
        score += 27
    elif row["credit_score"] >= 700:
        score += 24
    elif row["credit_score"] >= 650:
        score += 18
    elif row["credit_score"] >= 600:
        score += 10
        reasons.append("Average Credit Score")
    elif row["credit_score"] >= 550:
        score += 5
        reasons.append("Low Credit Score")
    else:
        reasons.append("Very Low Credit Score")

    # -------------------------------------------------------
    # 2. Debt To Income Ratio (20 Points)
    # -------------------------------------------------------
    dti = row["debt_to_income_ratio"]

    if dti < 20:
        score += 20
    elif dti < 30:
        score += 18
    elif dti < 40:
        score += 15
    elif dti < 50:
        score += 10
        reasons.append("Moderate DTI")
    elif dti < 60:
        score += 5
        reasons.append("High DTI")
    else:
        reasons.append("Very High DTI")

    # -------------------------------------------------------
    # 3. Late Payments (15 Points)
    # -------------------------------------------------------
    # lp = row["LatePaymentsLastYear"]

    # if lp == 0:
    #     score += 15
    # elif lp == 1:
    #     score += 12
    # elif lp == 2:
    #     score += 8
    #     reasons.append("2 Late Payments")
    # elif lp == 3:
    #     score += 4
    #     reasons.append("Frequent Late Payments")
    # else:
    #     reasons.append("Too Many Late Payments")

    # -------------------------------------------------------
    # 4. Annual Income (10 Points)
    # -------------------------------------------------------
    income = row["annual_income"]

    if income >= 5000000:
        score += 10
    elif income >= 3000000:
        score += 8
    elif income >= 2000000:
        score += 6
    elif income >= 1000000:
        score += 4
    else:
        score += 2
        reasons.append("Low Income")

    # -------------------------------------------------------
    # 5. Loan To Income Ratio - DTI (8 Points)
    # -------------------------------------------------------
    ratio = row["debt_to_income_ratio"] / max(income, 1)

    if ratio < 0.20:
        score += 8
    elif ratio < 0.40:
        score += 6
    elif ratio < 0.60:
        score += 4
    elif ratio < 0.80:
        score += 2
    else:
        reasons.append("Loan Amount Too High")

    # -------------------------------------------------------
    # 6. Employment Status (5 Points)
    # -------------------------------------------------------
    emp = row["employment_status"]

    if emp in ["Government", "Salaried"]:
        score += 5
    elif emp == "Business":
        score += 4
    elif emp == "Self-Employed":
        score += 3
    elif emp == "Contract":
        score += 2
    else:
        reasons.append("Employment Risk")

    # -------------------------------------------------------
    # 7. Years At Current Job (4 Points)
    # -------------------------------------------------------
    years = row["employment_years"]

    if years >= 10:
        score += 4
    elif years >= 5:
        score += 3
    elif years >= 2:
        score += 2
    elif years >= 1:
        score += 1
    else:
        reasons.append("New Employee")

    # -------------------------------------------------------
    # 8. Existing Loans (3 Points)
    # -------------------------------------------------------
    loans = row["existing_loans_count"]

    if loans == 0:
        score += 3
    elif loans == 1:
        score += 2
    elif loans == 2:
        score += 1
    else:
        reasons.append("Too Many Existing Loans")

    # -------------------------------------------------------
    # 9. Credit Utilization (3 Points)
    # -------------------------------------------------------
    # util = row["CreditUtilization"]

    # if util < 30:
    #     score += 3
    # elif util < 50:
    #     score += 2
    # elif util < 75:
    #     score += 1
    # else:
    #     reasons.append("High Credit Utilization")

    # -------------------------------------------------------
    # 10. Savings Balance (2 Points)
    # -------------------------------------------------------
    savings = row["savings_balance"]

    if savings >= 1000000:
        score += 2
    elif savings >= 250000:
        score += 1
    else:
        reasons.append("Low Savings")

    # -------------------------------------------------------
    # 11. Previous Default (2 Points)
    # -------------------------------------------------------
    
    if row["previous_default"] == 0:
        score += 15
    else:
        reasons.append("Previous Loan Default")
    
    # -------------------------------------------------------
    # Risk Level
    # -------------------------------------------------------
    if score >= 85:
        risk = "Very Low Risk"
        approval = "Approved"

    elif score >= 70:
        risk = "Low Risk"
        approval = "Approved"

    elif score >= 55:
        risk = "Medium Risk"
        approval = "Manual Review"

    elif score >= 40:
        risk = "High Risk"
        approval = "Rejected"

    else:
        risk = "Very High Risk"
        approval = "Rejected"

    if len(reasons) == 0:
        reasons.append("Excellent Applicant")

    return pd.Series([
        score,
        risk,
        approval,
        ", ".join(reasons)
    ])

In [13]:
# First update/create the columns


# Convert actual labels to Yes/No for reporting
df["loanapproval"] = df["loanapproval"].map({
    0: "No",
    1: "Yes"
})

df["AI_Prediction"] = pd.Series(
    pipeline.predict(X),
    index=df.index
).map({0: "Rejected", 1: "Approved"})

df["AI_Probability"] = (
    pipeline.predict_proba(X)[:, 1] * 100
).round(2)

df[[
    "RiskScore",
    "RiskLevel",
    "ApprovalStatus",
    "ApprovalRejectReason"
]] = df.apply(calculate_loan_decision, axis=1)

# ------------------------------------
# Reorder columns
# ------------------------------------

cols = df.columns.tolist()

# Remove the columns to be repositioned
for c in [
    "AI_Prediction",
    "AI_Probability",
    "RiskScore",
    "RiskLevel",
    "ApprovalStatus",
    "ApprovalRejectReason",
]:
    cols.remove(c)

# Find position of loanapproval
idx = cols.index("loanapproval") + 1

# Insert them in the desired order
new_cols = (
    cols[:idx]
    + [
        "AI_Prediction",
        "AI_Probability",
        "RiskScore",
        "RiskLevel",
        "ApprovalStatus",
        "ApprovalRejectReason",
    ]
    + cols[idx:]
)

df = df[new_cols]

df

,loan_id,age,gender,marital_status,dependents,education,employment_status,employment_years,annual_income,credit_score,...,savings_balance,collateral_value,debt_to_income_ratio,loanapproval,AI_Prediction,AI_Probability,RiskScore,RiskLevel,ApprovalStatus,ApprovalRejectReason
0,LN100000,43,Male,Divorced,1,High School,Self-Employed,21.5,16354,595,...,3079,4856,0.812,No,Rejected,27.590000,58,Medium Risk,Manual Review,"Low Credit Score, Low Income, Low Savings"
1,LN100001,36,Female,Married,2,Bachelor's,Employed,9.6,59460,579,...,1758,155688,0.678,No,Rejected,10.610000,56,Medium Risk,Manual Review,"Low Credit Score, Low Income, Employment Risk,..."
2,LN100002,45,Male,Married,1,Master's,Self-Employed,7.2,89708,574,...,2280,33927,0.037,Yes,Approved,86.349998,58,Medium Risk,Manual Review,"Low Credit Score, Low Income, Low Savings"
3,LN100003,54,Male,Single,1,Bachelor's,Employed,13.8,25560,550,...,3249,0,0.328,No,Rejected,24.459999,40,High Risk,Rejected,"Low Credit Score, Low Income, Employment Risk,..."
4,LN100004,35,Female,Single,0,Bachelor's,Employed,4.5,54248,683,...,5857,16828,0.062,Yes,Approved,99.489998,68,Medium Risk,Manual Review,"Low Income, Employment Risk, Low Savings"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10495,LN110495,35,Female,Single,3,Bachelor's,Retired,2.7,22904,498,...,2642,18651,0.290,No,Rejected,6.450000,47,High Risk,Rejected,"Very Low Credit Score, Low Income, Employment ..."
10496,LN110496,21,Male,Married,0,Bachelor's,Employed,0.8,59789,550,...,1236,14075,0.125,Yes,Approved,58.770000,50,High Risk,Rejected,"Low Credit Score, Low Income, Employment Risk,..."
10497,LN110497,38,Female,Married,0,Bachelor's,Employed,16.2,63805,480,...,2300,65116,0.213,No,Rejected,6.130000,49,High Risk,Rejected,"Very Low Credit Score, Low Income, Employment ..."
10498,LN110498,51,Male,Married,0,Bachelor's,Retired,26.1,104933,595,...,4255,0,0.061,Yes,Approved,96.690002,57,Medium Risk,Manual Review,"Low Credit Score, Low Income, Employment Risk,..."


In [14]:
df.to_excel(
    "Loan_Risk_Assessment_AI_BusinessLogic.xlsx",
    index=False
)

print("Loan Risk Assessment Report Exported Successfully")

Loan Risk Assessment Report Exported Successfully


In [16]:
import joblib

joblib.dump(pipeline, "Loan_Risk_Calculator.pkl")
print("Model saved successfully")



Model saved successfully
